# DAKGEA - Data Augmentation con PLM

In questo notebook aggiungeremo **data augmentation** all'esperimento precedente.

L'augmentation usa un **Pre-trained Language Model (PLM)** per generare entità sintetiche che migliorano l'allineamento.

## Setup

In [ ]:
import sys
from pathlib import Path
import yaml
import json

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from experiments.runner.runner import ExperimentRunner
from src.config.loader import load_yaml

## Configurazione con Augmentation

Rispetto al notebook precedente, aggiungiamo la sezione **augmentation**.

In [ ]:
config = {
    "experiment": {
        "name": "tutorial_augmented",
        
        # Dataset
        "dataset": {
            "name": "openea/D_W_15K_V1",
            "writer": "bert_int"
        },
        
        # Reduction: 30% delle entità
        "reduction": {
            "method": "random_entities",
            "ratio": 0.3,
            "writer": "bert_int",
            "eval": True,
            "save_dataset": False,
            "save_model": False
        },
        
        # NUOVA SEZIONE: Augmentation
        "augmentation": {
            "method": "plm",                # Usa PLM (BART)
            "ratio": 0.5,                   # Genera 50% entità in più
            "writer": {
                "type": "bert_int",
                "augmented_only_train": True  # Entità sintetiche solo in training
            },
            "eval": True,                   # Valuta anche con augmentation
            "save_dataset": False,
            "save_model": False
        },
        
        "model": "bert_int",
        "seed": 42,
        "clear": True,
        "overwrite_existing": False
    }
}

print(yaml.dump(config, default_flow_style=False))

## Come Funziona l'Augmentation

1. **Reduction**: Seleziona 30% delle entità originali
2. **Augmentation**: 
   - Analizza le entità ridotte
   - Usa BART (PLM) per generare entità sintetiche simili
   - Genera il 50% in più di dati (ratio=0.5)
3. **Training**: 
   - Training set: entità originali + sintetiche
   - Test set: solo entità originali
4. **Evaluation**: Valuta su entità reali

In [ ]:
# Salva la configurazione
config_file = PROJECT_ROOT / "config" / "experiments" / "tutorial_augmented.yaml"

with open(config_file, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"✓ Config saved to: {config_file}")

## Eseguire l'Esperimento

⚠️ **Nota**: L'augmentation richiede più tempo del baseline (10-20 minuti)

In [ ]:
# Carica e crea il runner
config_data = load_yaml(config_file)
runner = ExperimentRunner(
    payload=config_data,
    cli_overwrite=False,
    default_overwrite=False
)

print(f"Experiment: {runner.name}")
print(f"Stages: {len(runner.augmentations) > 0 and 'reduction + augmentation' or 'reduction only'}")

In [ ]:
# ESEGUI
print("Starting augmented experiment...")
print("This will take 10-20 minutes.\n")

try:
    runner.run()
    print("\n✓ Experiment completed!")
except Exception as e:
    print(f"\n✗ Failed: {e}")
    import traceback
    traceback.print_exc()

## Confronto Baseline vs Augmented

In [ ]:
# Carica risultati baseline (dal notebook precedente)
baseline_file = PROJECT_ROOT / "results" / "tutorial_baseline" / "reduction_030" / "D_W_15K_V1" / "reduction" / "results.json"

# Carica risultati augmented
augmented_file = runner.base_workspace / "reduction_030" / "D_W_15K_V1" / "augmentation" / "results.json"

def load_results(file_path):
    if file_path.exists():
        with open(file_path) as f:
            return json.load(f)
    return None

baseline_results = load_results(baseline_file)
augmented_results = load_results(augmented_file)

if baseline_results and augmented_results:
    print("=" * 80)
    print("CONFRONTO: Baseline vs Augmented")
    print("=" * 80)
    
    model = "bert_int"
    baseline = baseline_results.get(model, {})
    augmented = augmented_results.get(model, {})
    
    metrics = ["hits@1", "hits@5", "hits@10", "mrr", "precision", "recall", "f-measure"]
    
    print(f"\n{'Metric':<15} {'Baseline':<12} {'Augmented':<12} {'Δ':<12} {'Improvement'}")
    print("-" * 80)
    
    for metric in metrics:
        base_val = baseline.get(metric, 0)
        aug_val = augmented.get(metric, 0)
        delta = aug_val - base_val
        improvement = (delta / base_val * 100) if base_val > 0 else 0
        
        symbol = "✓" if delta > 0 else "✗" if delta < 0 else "="
        
        print(f"{metric:<15} {base_val:<12.4f} {aug_val:<12.4f} {delta:+.4f}      {symbol} {improvement:+.2f}%")
    
    print("\n" + "=" * 80)
    print("Legenda:")
    print("  ✓ = Miglioramento")
    print("  ✗ = Peggioramento")
    print("  = = Nessun cambiamento")
else:
    if not baseline_results:
        print("⚠ Baseline results not found. Run notebook 02 first.")
    if not augmented_results:
        print("⚠ Augmented results not found. Check if experiment completed successfully.")

## Interpretazione dei Risultati

### Cosa Aspettarsi

- **Hits@1 migliora**: L'augmentation aumenta la precisione top-1
- **Recall migliora**: Più dati sintetici aiutano a trovare più allineamenti
- **Il miglioramento varia**: Dipende dal dataset e dai parametri

### Parametri da Tuning

- **reduction.ratio**: Quante entità originali usare (0.1 - 1.0)
- **augmentation.ratio**: Quante entità sintetiche generare (0.1 - 1.0)
- **augmentation.method**: Quale metodo usare (plm, stub, ecc.)

### Combinazioni Comuni

```yaml
# Pochi dati, molta augmentation
reduction.ratio: 0.1    # 10% dati originali
augmentation.ratio: 1.0  # 100% dati sintetici

# Dati bilanciati
reduction.ratio: 0.3    # 30% dati originali
augmentation.ratio: 0.5  # 50% dati sintetici

# Molti dati, poca augmentation
reduction.ratio: 0.8    # 80% dati originali
augmentation.ratio: 0.2  # 20% dati sintetici
```

## Analisi Dettagliata dei Dati Generati

Possiamo ispezionare le entità sintetiche generate:

In [ ]:
# Carica il summary dell'augmentation
summary_file = runner.base_workspace / "reduction_030" / "D_W_15K_V1" / "augmentation" / "summary.json"

if summary_file.exists():
    with open(summary_file) as f:
        summary = json.load(f)
    
    print("Augmentation Summary:")
    print("-" * 40)
    print(f"Method: {summary.get('method', 'N/A')}")
    print(f"Ratio: {summary.get('ratio', 'N/A')}")
    print(f"Original pairs: {summary.get('original_pairs', 'N/A')}")
    print(f"Synthetic pairs: {summary.get('synthetic_pairs', 'N/A')}")
    print(f"Total pairs: {summary.get('total_pairs', 'N/A')}")
else:
    print("Summary file not found")

## Prossimi Passi

Nel prossimo notebook vedremo:
- Come analizzare i risultati in dettaglio
- Come generare grafici e statistiche
- Come confrontare multiple configurazioni

➡️ Continua con: **04_analyze_results.ipynb**